# Identifying Diabetic Patients from Administrative Claims Data

**A reproducible, end-to-end case study in claims-based phenotyping**

This notebook accompanies the blog post of the same title. It walks through a production-style workflow for identifying diabetic patients from claims data using ICD-10-CM, RxNorm, LOINC, and CPT codes. Every cell is executable with only `pandas`, `numpy`, and (optionally) `scikit-learn`.

**Phenotype definition.** A patient is classified as diabetic if **any** of the following is true:

- **(A)** Two or more **outpatient** claims on *different dates* carrying an ICD-10-CM code in E10–E13, **or**
- **(B)** One or more **inpatient** claims with an ICD-10-CM code in E10–E13, **or**
- **(C)** One or more pharmacy claims for an antidiabetic agent identified by RxNorm.

The cohort is then enriched with LOINC-based HbA1c results and validated against CPT 83036 (A1c testing procedure).


## 1. Setup

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 120)

RANDOM_SEED = 42


## 2. Code Lists

Production deployments source these from authoritative value sets (e.g., NLM Value Set Authority Center, CMS Chronic Conditions Warehouse). Here we hard-code a minimal set for demonstration.

In [2]:
# ICD-10-CM diabetes chapter prefixes
ICD_DM_PREFIXES = ("E10", "E11", "E12", "E13")

# Antidiabetic agents (illustrative ingredient-level RxCUIs)
ANTIDM_RXCUIS = {
    6809:   "metformin",
    5856:   "insulin",
    593411: "sitagliptin",
    857974: "glipizide",
    1373458:"empagliflozin",
    1551291:"liraglutide",
    860975: "dulaglutide",
    200370: "glyburide",
}

# Non-antidiabetic drugs used to populate synthetic data
NON_DM_RXCUIS = {
    617312: "atorvastatin", 617314: "simvastatin", 197319: "amlodipine",
    310965: "lisinopril",   1044427:"gabapentin",  866924: "metoprolol",
}

# Hemoglobin A1c LOINC codes
A1C_LOINCS = {"4548-4", "17856-6", "41995-2"}

# CPT procedure code for HbA1c testing
CPT_A1C = "83036"


## 3. Generating a Realistic Synthetic Claims Dataset

Real claims data is proprietary. For reproducibility, we generate a synthetic dataset that mirrors CMS Limited Data Set structure:

| Table               | Grain                       | Key columns                                               |
|---------------------|-----------------------------|-----------------------------------------------------------|
| `patients`          | one row per patient         | `patient_id`, `age`, `sex`                                |
| `diagnosis_claims`  | one row per diagnosis line  | `patient_id`, `service_date`, `claim_type`, `icd_code`    |
| `pharmacy_claims`   | one row per Rx fill         | `patient_id`, `service_date`, `rxnorm_code`, `drug_name`  |
| `lab_results`       | one row per lab observation | `patient_id`, `service_date`, `loinc_code`, `value`       |
| `procedure_claims`  | one row per procedure line  | `patient_id`, `service_date`, `cpt_code`                  |

To evaluate phenotype performance we also track a `is_diabetic_truth` column.

In [3]:
def generate_synthetic_data(n_patients: int = 5000, seed: int = RANDOM_SEED):
    rng = np.random.default_rng(seed)

    patients = pd.DataFrame({
        "patient_id": [f"P{i:06d}" for i in range(n_patients)],
        "age":        rng.integers(20, 85, n_patients),
        "sex":        rng.choice(["M", "F"], n_patients),
    })
    truth = rng.random(n_patients) < 0.22         # 22% true diabetic prevalence

    # ---- Diagnosis claims (vectorized) ----
    n_dx = rng.poisson(5, n_patients) + rng.poisson(4, n_patients) * truth.astype(int)
    pid_idx = np.repeat(np.arange(n_patients), n_dx)
    total = n_dx.sum()
    is_dm_claim = (rng.random(total) < 0.55) & truth[pid_idx]
    dm_codes     = ["E11.9", "E11.65", "E11.22", "E11.9", "E10.9", "E13.9"]
    non_dm_codes = ["I10", "M25.5", "R51", "J45.9", "Z00.00", "N39.0", "I25.10", "M54.5"]
    codes = np.where(is_dm_claim, rng.choice(dm_codes, total), rng.choice(non_dm_codes, total))
    # Simulate occasional erroneous diabetes coding on non-diabetics
    noise = (rng.random(total) < 0.015) & ~truth[pid_idx]
    codes = np.where(noise, "E11.9", codes)
    diagnosis_claims = pd.DataFrame({
        "patient_id":   patients["patient_id"].values[pid_idx],
        "claim_id":     [f"DX{i:09d}" for i in range(total)],
        "service_date": pd.to_datetime("2025-01-01") + pd.to_timedelta(rng.integers(0, 365, total), unit="D"),
        "claim_type":   rng.choice(["outpatient", "inpatient"], total, p=[0.88, 0.12]),
        "icd_code":     codes,
    })

    # ---- Pharmacy claims ----
    n_rx = rng.poisson(1, n_patients) + rng.poisson(4, n_patients) * truth.astype(int)
    pid_rx = np.repeat(np.arange(n_patients), n_rx)
    total_rx = n_rx.sum()
    is_antidm = (rng.random(total_rx) < 0.70) & truth[pid_rx]
    antidm_arr = np.array(list(ANTIDM_RXCUIS.keys()))
    non_arr    = np.array(list(NON_DM_RXCUIS.keys()))
    rxcui = np.where(is_antidm,
                     rng.choice(antidm_arr, total_rx),
                     rng.choice(non_arr, total_rx))
    name_map = {**ANTIDM_RXCUIS, **NON_DM_RXCUIS}
    pharmacy_claims = pd.DataFrame({
        "patient_id":   patients["patient_id"].values[pid_rx],
        "claim_id":     [f"RX{i:09d}" for i in range(total_rx)],
        "service_date": pd.to_datetime("2025-01-01") + pd.to_timedelta(rng.integers(0, 365, total_rx), unit="D"),
        "rxnorm_code":  rxcui,
        "drug_name":    [name_map[c] for c in rxcui],
    })

    # ---- Lab results (HbA1c) ----
    has_a1c = ((rng.random(n_patients) < 0.85) & truth) | (rng.random(n_patients) < 0.12)
    n_a1c = rng.integers(1, 4, n_patients) * has_a1c.astype(int)
    pid_lab = np.repeat(np.arange(n_patients), n_a1c)
    total_a1c = int(n_a1c.sum())
    values = np.where(truth[pid_lab],
                      rng.normal(8.0, 1.5, total_a1c),
                      rng.normal(5.5, 0.4, total_a1c)).clip(4.0, 14.0).round(1)
    lab_results = pd.DataFrame({
        "patient_id":   patients["patient_id"].values[pid_lab],
        "service_date": pd.to_datetime("2025-01-01") + pd.to_timedelta(rng.integers(0, 365, total_a1c), unit="D"),
        "loinc_code":   rng.choice(list(A1C_LOINCS), total_a1c),
        "test_name":    "Hemoglobin A1c",
        "value":        values,
        "unit":         "%",
    })

    # ---- Procedure claims ----
    a1c_proc = lab_results[["patient_id", "service_date"]].copy()
    a1c_proc["cpt_code"] = CPT_A1C
    a1c_proc["claim_id"] = [f"PR{i:09d}" for i in range(len(a1c_proc))]
    extra = n_patients * 3
    other_proc = pd.DataFrame({
        "patient_id":   rng.choice(patients["patient_id"], extra),
        "claim_id":     [f"PR{i:09d}" for i in range(len(a1c_proc), len(a1c_proc)+extra)],
        "service_date": pd.to_datetime("2025-01-01") + pd.to_timedelta(rng.integers(0, 365, extra), unit="D"),
        "cpt_code":     rng.choice(["99213", "99214", "80053", "85025", "93000"], extra),
    })
    procedure_claims = pd.concat([a1c_proc, other_proc], ignore_index=True)[
        ["patient_id", "claim_id", "service_date", "cpt_code"]
    ]

    truth_df = patients[["patient_id"]].copy()
    truth_df["is_diabetic_truth"] = truth
    return patients, truth_df, diagnosis_claims, pharmacy_claims, lab_results, procedure_claims


patients, truth, diagnosis_claims, pharmacy_claims, lab_results, procedure_claims = \
    generate_synthetic_data(n_patients=5000)

for name, df in [("patients", patients), ("diagnosis", diagnosis_claims),
                 ("pharmacy", pharmacy_claims), ("labs", lab_results),
                 ("procedures", procedure_claims)]:
    print(f"{name:>12}: {len(df):>8,} rows")


    patients:    5,000 rows
   diagnosis:   29,688 rows
    pharmacy:    9,520 rows
        labs:    2,916 rows
  procedures:   17,916 rows


## 4. Schema Normalization

Before any analytics, we force dates to `datetime64[ns]` and cast repeated string fields to `category`. On a 10M-row claims table this step alone can cut memory use by 5–10×.

In [4]:
def normalize_claims(df: pd.DataFrame, date_col: str = "service_date") -> pd.DataFrame:
    df = df.copy()
    if date_col in df.columns:
        df[date_col] = pd.to_datetime(df[date_col]).dt.normalize()
    for c in ("icd_code", "cpt_code", "loinc_code", "claim_type"):
        if c in df.columns:
            df[c] = df[c].astype("category")
    return df

diagnosis_claims  = normalize_claims(diagnosis_claims)
pharmacy_claims   = normalize_claims(pharmacy_claims)
lab_results       = normalize_claims(lab_results)
procedure_claims  = normalize_claims(procedure_claims)

print(diagnosis_claims.dtypes)


patient_id              object
claim_id                object
service_date    datetime64[ns]
claim_type            category
icd_code              category
dtype: object


## 5. Phenotype Logic

We translate the clinical definition into three independent, composable functions. Each returns the **set of patient_ids** that satisfy one criterion, making the final OR trivially expressible.

In [5]:
def filter_diabetes_diagnoses(df_dx: pd.DataFrame) -> pd.DataFrame:
    """Return only rows whose ICD code starts with E10–E13."""
    code = df_dx["icd_code"].astype(str)
    mask = code.str.startswith(ICD_DM_PREFIXES)
    return df_dx[mask]


def patients_with_outpatient_dx(df_dx_dm: pd.DataFrame,
                                min_distinct_dates: int = 2) -> set:
    """Criterion (A): ≥ N outpatient claims on distinct dates."""
    outp = df_dx_dm[df_dx_dm["claim_type"] == "outpatient"]
    distinct = outp.groupby("patient_id", observed=True)["service_date"].nunique()
    return set(distinct[distinct >= min_distinct_dates].index)


def patients_with_inpatient_dx(df_dx_dm: pd.DataFrame) -> set:
    """Criterion (B): any inpatient claim with a diabetes ICD code."""
    inp = df_dx_dm[df_dx_dm["claim_type"] == "inpatient"]
    return set(inp["patient_id"].unique())


def patients_with_antidm_rx(df_rx: pd.DataFrame) -> set:
    """Criterion (C): any pharmacy claim for an antidiabetic agent."""
    mask = df_rx["rxnorm_code"].isin(ANTIDM_RXCUIS.keys())
    return set(df_rx[mask]["patient_id"].unique())


dm_dx = filter_diabetes_diagnoses(diagnosis_claims)
print("Diabetes-coded diagnosis rows:", len(dm_dx))
print("Criterion A patients:", len(patients_with_outpatient_dx(dm_dx)))
print("Criterion B patients:", len(patients_with_inpatient_dx(dm_dx)))
print("Criterion C patients:", len(patients_with_antidm_rx(pharmacy_claims)))


Diabetes-coded diagnosis rows: 5786
Criterion A patients: 1040
Criterion B patients: 520
Criterion C patients: 1087


## 6. Cohort Construction

We attach per-criterion booleans to the patient table; `is_diabetic` is simply their logical OR. Storing the intermediate flags is a best practice for transparency and downstream analytics.

In [6]:
def build_cohort(patients: pd.DataFrame,
                 df_dx: pd.DataFrame,
                 df_rx: pd.DataFrame) -> pd.DataFrame:
    dm_dx = filter_diabetes_diagnoses(df_dx)
    outp  = patients_with_outpatient_dx(dm_dx)
    inp   = patients_with_inpatient_dx(dm_dx)
    rx    = patients_with_antidm_rx(df_rx)

    cohort = patients.copy()
    cohort["has_outpatient_dx"] = cohort["patient_id"].isin(outp)
    cohort["has_inpatient_dx"]  = cohort["patient_id"].isin(inp)
    cohort["has_antidm_rx"]     = cohort["patient_id"].isin(rx)
    cohort["is_diabetic"] = cohort[
        ["has_outpatient_dx", "has_inpatient_dx", "has_antidm_rx"]
    ].any(axis=1)
    return cohort


cohort = build_cohort(patients, diagnosis_claims, pharmacy_claims)
print("Cohort size:", cohort["is_diabetic"].sum(), "/", len(cohort))
cohort.head()


Cohort size: 1146 / 5000


,patient_id,age,sex,has_outpatient_dx,has_inpatient_dx,has_antidm_rx,is_diabetic
0,P000000,25,M,True,False,True,True
1,P000001,70,M,False,False,False,False
2,P000002,62,F,False,False,False,False
3,P000003,48,F,True,False,True,True
4,P000004,48,F,True,True,True,True


## 7. Lab Enrichment (LOINC → HbA1c)

Once identified, the cohort is enriched with A1c values: mean, max, latest, and count. Missing labs are left as `NaN` so downstream models can handle them explicitly.

In [7]:
def enrich_with_a1c(cohort: pd.DataFrame, df_lab: pd.DataFrame) -> pd.DataFrame:
    a1c = df_lab[df_lab["loinc_code"].isin(A1C_LOINCS)].copy()
    if a1c.empty:
        for col in ("latest_a1c", "mean_a1c", "max_a1c", "n_a1c", "latest_a1c_date"):
            cohort[col] = np.nan
        cohort["poor_control"] = False
        return cohort

    agg = a1c.groupby("patient_id").agg(
        mean_a1c        = ("value",        "mean"),
        max_a1c         = ("value",        "max"),
        n_a1c           = ("value",        "size"),
        latest_a1c_date = ("service_date", "max"),
    )
    a1c_sorted = a1c.sort_values(["patient_id", "service_date"])
    latest = (a1c_sorted.groupby("patient_id").tail(1)
                        .set_index("patient_id")["value"].rename("latest_a1c"))
    agg = agg.join(latest)

    out = cohort.merge(agg, on="patient_id", how="left")
    out["poor_control"] = out["latest_a1c"].fillna(0) > 8.0
    return out


cohort = enrich_with_a1c(cohort, lab_results)
cohort[cohort["is_diabetic"]][
    ["patient_id", "latest_a1c", "mean_a1c", "max_a1c", "n_a1c", "poor_control"]
].head()


,patient_id,latest_a1c,mean_a1c,max_a1c,n_a1c,poor_control
0,P000000,9.0,7.833333,9.0,3.0,True
3,P000003,NaN,NaN,NaN,NaN,False
4,P000004,6.9,6.900000,6.9,1.0,False
12,P000012,6.1,7.933333,10.4,3.0,False
14,P000014,6.8,6.750000,6.8,2.0,False


## 8. Procedure Validation (CPT 83036)

A positive A1c value is good evidence of diabetes, but it is additionally reassuring to confirm that the **procedure** of A1c testing was billed — this protects against value-level data errors.

In [8]:
def add_a1c_testing_flag(cohort: pd.DataFrame, df_proc: pd.DataFrame) -> pd.DataFrame:
    testers = set(df_proc.loc[df_proc["cpt_code"] == CPT_A1C, "patient_id"].unique())
    out = cohort.copy()
    out["has_a1c_testing"] = out["patient_id"].isin(testers)
    return out


cohort = add_a1c_testing_flag(cohort, procedure_claims)
cohort["has_a1c_testing"].value_counts()


has_a1c_testing
False    3557
True     1443
Name: count, dtype: int64

## 9. Feature Engineering

Good downstream models (cost, readmission, A1c trajectory) need more than a single boolean. We surface medication-class indicators and treatment-intensity metrics.

In [9]:
def engineer_features(cohort: pd.DataFrame, df_rx: pd.DataFrame) -> pd.DataFrame:
    dm_rx = df_rx[df_rx["rxnorm_code"].isin(ANTIDM_RXCUIS.keys())]
    n_agents = dm_rx.groupby("patient_id")["rxnorm_code"].nunique()
    insulin  = set(dm_rx.loc[dm_rx["rxnorm_code"] == 5856,   "patient_id"])
    met      = set(dm_rx.loc[dm_rx["rxnorm_code"] == 6809,   "patient_id"])
    glp1     = set(dm_rx.loc[dm_rx["rxnorm_code"].isin({1551291, 860975}), "patient_id"])
    sglt2    = set(dm_rx.loc[dm_rx["rxnorm_code"] == 1373458,"patient_id"])

    out = cohort.copy()
    out["n_antidm_agents"] = out["patient_id"].map(n_agents).fillna(0).astype(int)
    out["on_insulin"]   = out["patient_id"].isin(insulin)
    out["on_metformin"] = out["patient_id"].isin(met)
    out["on_glp1"]      = out["patient_id"].isin(glp1)
    out["on_sglt2"]     = out["patient_id"].isin(sglt2)
    return out


cohort = engineer_features(cohort, pharmacy_claims)
cohort[cohort["is_diabetic"]].head()


,patient_id,age,sex,has_outpatient_dx,has_inpatient_dx,has_antidm_rx,is_diabetic,mean_a1c,max_a1c,n_a1c,latest_a1c_date,latest_a1c,poor_control,has_a1c_testing,n_antidm_agents,on_insulin,on_metformin,on_glp1,on_sglt2
0,P000000,25,M,True,False,True,True,7.833333,9.0,3.0,2025-11-14,9.0,True,True,3,False,False,True,False
3,P000003,48,F,True,False,True,True,NaN,NaN,NaN,NaT,NaN,False,False,1,False,False,False,False
4,P000004,48,F,True,True,True,True,6.900000,6.9,1.0,2025-03-18,6.9,False,True,2,True,False,False,False
12,P000012,67,F,True,True,True,True,7.933333,10.4,3.0,2025-06-25,6.1,False,True,2,True,False,False,False
14,P000014,66,F,True,True,True,True,6.750000,6.8,2.0,2025-05-13,6.8,False,True,3,True,False,True,True


## 10. Phenotype Validation

Because our synthetic data carries ground-truth labels, we can measure the sensitivity, specificity, and positive predictive value of the phenotype itself.

In [10]:
eval_df = cohort.merge(truth, on="patient_id")
tp = ((eval_df.is_diabetic)  & (eval_df.is_diabetic_truth)).sum()
fp = ((eval_df.is_diabetic)  & (~eval_df.is_diabetic_truth)).sum()
fn = ((~eval_df.is_diabetic) & (eval_df.is_diabetic_truth)).sum()
tn = ((~eval_df.is_diabetic) & (~eval_df.is_diabetic_truth)).sum()
sens = tp / (tp + fn) if (tp + fn) else 0
spec = tn / (tn + fp) if (tn + fp) else 0
ppv  = tp / (tp + fp) if (tp + fp) else 0
print(f"Sensitivity = {sens:.3f}")
print(f"Specificity = {spec:.3f}")
print(f"PPV         = {ppv:.3f}")
print(f"Confusion matrix:\n  TP={tp}  FP={fp}\n  FN={fn}  TN={tn}")


Sensitivity = 1.000
Specificity = 0.990
PPV         = 0.966
Confusion matrix:
  TP=1107  FP=39
  FN=0  TN=3854


## 11. Scaling to Millions of Rows

Claims feeds from a regional payer typically contain 50–500M lines per year. `pandas` alone struggles past a few tens of millions. Three patterns cover most real-world needs.

### 11.1 Chunked Parquet streaming

If the data already sits in partitioned Parquet, you can stream row-groups and aggregate patient-level state incrementally. This keeps memory bounded regardless of dataset size.

In [11]:
from collections import defaultdict

def streaming_diabetes_flag(parquet_paths, chunk_size: int = 1_000_000):
    """Illustrative chunked aggregator for very large claims feeds."""
    outpatient_dates = defaultdict(set)
    inpatient_pids   = set()
    for path in parquet_paths:
        df = pd.read_parquet(path)                       # one shard at a time
        for start in range(0, len(df), chunk_size):
            chunk = df.iloc[start:start+chunk_size]
            dm = filter_diabetes_diagnoses(chunk)
            outp = dm[dm["claim_type"] == "outpatient"]
            for pid, s in outp.groupby("patient_id")["service_date"]:
                outpatient_dates[pid].update(s)
            inpatient_pids.update(
                dm.loc[dm["claim_type"] == "inpatient", "patient_id"].unique()
            )
    return (
        {pid for pid, dates in outpatient_dates.items() if len(dates) >= 2}
        | inpatient_pids
    )

# Example (not executed): streaming_diabetes_flag(["claims/year=2024/part-*.parquet"])


### 11.2 PySpark (optional)

For horizontally partitioned data, the same logic expresses cleanly in Spark SQL. The snippet below is illustrative — it does not run unless a Spark cluster is available.

In [12]:
# from pyspark.sql import SparkSession, functions as F
#
# spark = SparkSession.builder.getOrCreate()
# dx = spark.read.parquet("s3://bucket/claims/diagnosis/")
# dm = dx.filter(F.col("icd_code").rlike("^E1[0-3]"))
# outp = (dm.filter(F.col("claim_type") == "outpatient")
#           .groupBy("patient_id")
#           .agg(F.countDistinct("service_date").alias("n_dates"))
#           .filter(F.col("n_dates") >= 2))
# inp  = (dm.filter(F.col("claim_type") == "inpatient")
#           .select("patient_id").distinct())
# diabetic = outp.select("patient_id").union(inp).distinct()


### 11.3 Dask (optional)

Dask provides a drop-in DataFrame API over partitioned data. Only the partitioned read and the final `.compute()` differ from the pandas code.

In [13]:
# import dask.dataframe as dd
# dx = dd.read_parquet("s3://bucket/claims/diagnosis/")
# dm = dx[dx["icd_code"].str.startswith(ICD_DM_PREFIXES)]
# outp = (dm[dm["claim_type"] == "outpatient"]
#           .groupby("patient_id")["service_date"].nunique()
#           .compute())
# cohort_pids_A = set(outp[outp >= 2].index)


## 12. Bonus: Risk Stratification within the Diabetic Cohort

Once the cohort exists, code-derived features feed directly into predictive models. Below we train a simple logistic regression to predict **poor glycemic control** (latest A1c > 9.0) from age, treatment intensity, and medication-class flags.

> In production you would split by patient, respect temporal ordering, and use calibrated gradient-boosted trees (e.g., LightGBM) with HCC-style feature groupings.

In [14]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

dm_pop = cohort[cohort["is_diabetic"]].copy()
feature_cols = ["age", "n_antidm_agents", "on_insulin", "on_metformin",
                "on_glp1", "on_sglt2", "mean_a1c"]
dm_pop = dm_pop.dropna(subset=["mean_a1c"])
X = dm_pop[feature_cols].astype(float).fillna(0)
y = (dm_pop["latest_a1c"] > 9.0).astype(int)

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=RANDOM_SEED, stratify=y)
model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
auc = roc_auc_score(y_te, model.predict_proba(X_te)[:, 1])
print(f"Poor-control AUROC: {auc:.3f}")
print("\nCoefficients:")
print(pd.Series(model.coef_[0], index=feature_cols).sort_values(ascending=False))


Poor-control AUROC: 0.878

Coefficients:
mean_a1c           1.910586
on_metformin       0.307408
on_insulin         0.208818
on_sglt2           0.168006
age               -0.005638
n_antidm_agents   -0.071784
on_glp1           -0.086272
dtype: float64


## 13. Validation & Pitfalls

| Pitfall                                 | Mitigation                                                                           |
|-----------------------------------------|--------------------------------------------------------------------------------------|
| **False positives** from single rule-out codes | Require ≥2 outpatient dates *or* inpatient confirmation (already in phenotype)       |
| **Coding variability** between sites    | Use value sets from NLM VSAC; roll up to three-digit ICD prefixes when comparing     |
| **Missing labs**                        | Keep `has_a1c_testing` flag; impute with class-conditional median for modeling       |
| **Short lookback**                      | Run sensitivity analyses with 6-, 12-, and 24-month windows                          |
| **Rx without diagnosis**                | Metformin is sometimes prescribed for PCOS; require confirmation via Dx or A1c       |
| **Transition from ICD-9 to ICD-10**     | Break longitudinal studies at Oct 1, 2015 or use GEMs crosswalks                     |
| **Upcoding / payer mix**                | Validate phenotype across subgroups; report per-site performance                     |


## 14. Wrap-up

Starting from raw claims, we have:

1. Generated a reproducible synthetic dataset mirroring CMS table structure.
2. Normalized the schema.
3. Implemented the three-criterion phenotype as three small, testable functions.
4. Enriched the cohort with LOINC-based HbA1c values.
5. Validated via CPT 83036.
6. Engineered a compact feature set suitable for risk-adjustment or predictive modeling.
7. Sketched three strategies for scaling to production volumes.

The functions defined here (`filter_diabetes_diagnoses`, `build_cohort`, `enrich_with_a1c`, `engineer_features`) are deliberately small and composable — drop them into any claims pipeline with minimal glue.